In [ ]:
import pandas as pd
import numpy as np
import re

# =========================
# 1) LOAD DATA
# =========================
file_path = r"C:\Users\user\OneDrive - Helwan National University\Desktop\NN\dataset\tmdb_4genres_2500_each_final_updated.csv"
df = pd.read_csv(file_path)



# =========================
# 3) BASIC CLEANING
# =========================
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

df["tmdb_id"] = df["tmdb_id"].apply(safe_str)
df["overview"] = df["overview"].apply(safe_str)
df["label"] = df["label"].apply(safe_str)

df = df.drop_duplicates(subset=["tmdb_id"]).copy()

# =========================
# 4) CLEAN OVERVIEW
# =========================
def clean_overview(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["overview"] = df["overview"].apply(clean_overview)

# remove empty overviews
df = df[df["overview"] != ""].copy()

# =========================
# 5) CLEAN LABEL
# =========================
TARGET_GENRES = ["Action", "Horror", "Comedy", "Romance"]

df["label"] = df["label"].str.title()
df = df[df["label"].isin(TARGET_GENRES)].copy()

# =========================
# 6) FILTER SHORT TEXTS
# =========================
df["overview_words"] = df["overview"].apply(lambda x: len(x.split()))
df = df[df["overview_words"] >= 18].copy()

# =========================
# 7) FINAL CLEAN DATA
# =========================
df_final = df[["tmdb_id", "overview", "label"]].copy()

# =========================
# 8) SAVE
# =========================
df_final.to_csv("tmdb_overview_id_label_clean.csv", index=False, encoding="utf-8-sig")

print("Done:", df_final.shape)
print(df_final.head())

Done: (9996, 3)
  tmdb_id                                           overview   label
0   59709  it was the sister ship of the infamous titanic...  Action
1   84952  mr ozaki a japanese conglomerate businessman l...  Action
2   53461  this frenetic documentary about motorcycle dar...  Action
3   82384  high noon tells the story of a lawman named wi...  Action
4   11398  neil shaw is both agent and weapon a critical ...  Action


In [25]:
# نجيب أقل عدد موجود بين الأنواع
min_count = df_balanced["label"].value_counts().min()

balanced_parts = []

for label_name, group in df_balanced.groupby("label"):
    sampled_group = group.sample(n=min_count, random_state=42)
    balanced_parts.append(sampled_group)

df_final_balanced = pd.concat(balanced_parts, ignore_index=True)

print(df_final_balanced.columns.tolist())
print(df_final_balanced["label"].value_counts())
print(df_final_balanced.shape)

df_final_balanced.to_csv("tmdb_final_fully_balanced.csv", index=False, encoding="utf-8-sig")
print("Saved: tmdb_final_fully_balanced.csv")

['tmdb_id', 'title', 'original_title', 'release_date', 'release_year', 'original_language', 'overview', 'text_len_words', 'genres', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'runtime', 'adult', 'status', 'poster_path', 'backdrop_path', 'poster_url', 'poster_url_hd', 'backdrop_url', 'backdrop_url_hd', 'homepage', 'imdb_id', 'label', 'quality_score', 'overview_clean', 'overview_words_clean', 'overview_enhanced', 'overview_enhanced_words']
label
Action     2032
Comedy     2032
Horror     2032
Romance    2032
Name: count, dtype: int64
(8128, 30)
Saved: tmdb_final_fully_balanced.csv


In [24]:
import pandas as pd

df = pd.read_csv(r"dataset/tmdb_final_fully_balanced.csv")
print(df.head())

# الأعمدة اللي عايزة تسيبيها
keep_cols = [
    "overview_enhanced",
    "label",
    "tmdb_id",
    "title",
    "poster_path",
   
]

# نتأكد ناخد الموجود فقط
existing_cols = [col for col in keep_cols if col in df.columns]
missing_cols = [col for col in keep_cols if col not in df.columns]

print("Existing columns:")
print(existing_cols)

if missing_cols:
    print("\nMissing columns:")
    print(missing_cols)

# سيب الأعمدة المطلوبة فقط
df = df[existing_cols].copy()

# احذف الصفوف اللي ناقص فيها النص أو اللابل
df = df.dropna(subset=["overview_enhanced", "label"]).reset_index(drop=True)

# احفظ الملف الجديد
output_file = "tmdb_text_image_only.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nShape:", df.shape)
print("\nSaved as:", output_file)
print(df.head())

   tmdb_id                             title  \
0    68902            Shaolin: Wheel of Life   
1  1096563                    End of Loyalty   
2  1102332                        Jack's Law   
3    89828  Gang of Roses 2: Next Generation   
4    19206                 The Perfect Sleep   

                     original_title release_date  release_year  \
0            Shaolin: Wheel of Life   2001-12-11          2001   
1                    End of Loyalty   2023-03-07          2023   
2                        Jack's Law   2006-06-06          2006   
3  Gang of Roses 2: Next Generation   2012-02-10          2012   
4                 The Perfect Sleep   2009-03-13          2009   

  original_language                                           overview  \
0                en  Have you ever done a handstand... on the tips ...   
1                en  When the head of a notorious crime family is m...   
2                en  Jack Santos is a retired undercover vice cop w...   
3                e